In [ ]:
from pathlib import Path

import numpy as np
import pandas
import rasterio

In [ ]:
base_path = Path("../../../processed_data/nbs-river-catchment")

In [ ]:
def sum_tif(path):
    with rasterio.open(path) as src:
        data = src.read(1)

    return np.nan_to_num(data).sum()

In [ ]:
dr_max = sum_tif(base_path / "damage_reduction_max.tif")
dr_min = sum_tif(base_path / "damage_reduction_min.tif")
ar_max = sum_tif(base_path / "avoided__fluvial__ead_max.tif")
ar_min = sum_tif(base_path / "avoided__fluvial__ead_min.tif")

In [ ]:
JD_TO_USD = 1 / 150
MILLIONS = 1e-6
ENSEMBLE_INFO = {
    "7": "max",
    "10": "min"
}

In [ ]:
ar_min * JD_TO_USD * MILLIONS, dr_min * JD_TO_USD * MILLIONS

In [ ]:
(ar_min - dr_min) * JD_TO_USD

In [ ]:
ar_max * JD_TO_USD * MILLIONS, dr_max * JD_TO_USD * MILLIONS

In [ ]:
(ar_max - dr_max) * JD_TO_USD

In [ ]:
future_damage = pandas.read_parquet(base_path / "damage__future.parquet")

In [ ]:
future_damage["avoided__fluvial__ead"] = future_damage.baseline__fluvial__ead - future_damage.future__fluvial__ead

In [ ]:
df_min, df_max = 0, 0
for eid, subset in future_damage.groupby("ensemble_member"):
    if eid == "10":
        print(eid, "is min ensemble member")
        df_min = subset.avoided__fluvial__ead.sum()
    else:
        print(eid, "is max ensemble member")
        df_max = subset.avoided__fluvial__ead.sum()

df_min * JD_TO_USD * MILLIONS, df_max  * JD_TO_USD * MILLIONS

In [ ]:
(ar_min - df_min)* JD_TO_USD

In [ ]:
(ar_max - df_max)* JD_TO_USD

In [ ]:
JD_TO_USD = 1 / 150
MILLIONS = 1e-6
ENSEMBLE_INFO = {
    "7": "max",
    "10": "min"
}
for eid, subset in future_damage.groupby("ensemble_member"):
    print(ENSEMBLE_INFO[eid], subset.avoided__fluvial__ead.sum() * JD_TO_USD * MILLIONS, "$US million")

In [ ]:
(df_min + df_max ) * JD_TO_USD * MILLIONS